### [3강] 따릉이는 어디까지 닿고 있는가  
#### Folium 기반 따릉이 커버리지 시각화

3강에서는 2강에서 도출한 교통 비커버 지역을 바탕으로, **따릉이가 해당 지역들을 얼마나 보완적으로 커버하고 있는지**를 Folium 지도 위에서 직관적으로 확인한다.

이를 위해 본 강의에서는 GeoPandas로 전처리한 공간 데이터를 Folium으로 시각화하고,  
OSM 도로 네트워크를 활용한 **경로 계산 라이브러리**를 통해 따릉이 대여소를 중심으로 한 실제 이동 가능 범위를 함께 고려한다.

구체적으로는, OSMnx를 이용해 보행·자전거 도로 네트워크를 구성하고, NetworkX 기반 최단경로 알고리즘을 적용하여  
단순 직선 거리 대신 **현실적인 이동 경로 관점**에서 따릉이의 공간적 커버 구조를 해석한다.

행정경계, 비커버 폴리곤, 따릉이 대여소, 버스정류장, 지하철역을 하나의 인터랙티브 지도에 통합함으로써,  
따릉이가 교통 비커버 공간을 어느 정도 보완하고 있는지와 여전히 남아 있는 접근 사각지대를 시각적으로 파악하는 것이 본 강의의 핵심 목표이다.

본 강의는 지표 계산보다는 **공간 분포와 실제 이동 구조를 함께 이해하는 단계**로서, 이후 경로 기반 개선 시나리오 분석으로 확장하기 위한 기반을 제공한다.

- 서울시 따릉이 데이터 : https://data.seoul.go.kr/dataList/OA-21235/S/1/datasetView.do
- 버스정류장 위치정보 데이터 : https://data.seoul.go.kr/dataList/OA-15067/S/1/datasetView.do
- 지하철역 위치정보 데이터 : https://www.data.go.kr/data/15099316/fileData.do 
- 행정동 벡터 데이터 : https://www.vworld.kr/dtmk/dtmk_ntads_s002.do?svcCde=MK&dsId=30017

- Folium documentation : https://python-visualization.github.io/folium/latest/

In [26]:
%pip install osmnx folium geopandas shapely networkx openpyxl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, warnings # 파일 경로 처리(os)와 경고 메시지 제어(warnings)를 위해 표준 라이브러리 임포트
warnings.filterwarnings("ignore") # 실행 중 발생하는 경고(warning) 메시지를 화면에 출력하지 않도록 설정

import numpy as np # 수치 계산 및 배열 연산을 위한 NumPy 라이브러리 임포트 (별칭: np)
import pandas as pd # 테이블 형태의 데이터 처리(DataFrame)를 위한 Pandas 라이브러리 임포트 (별칭: pd)

import geopandas as gpd # 공간 데이터(Geometry 포함 DataFrame) 처리를 위한 GeoPandas 임포트 (별칭: gpd)
import folium # Leaflet 기반의 인터랙티브 웹 지도 생성을 위한 Folium 라이브러리 임포트
from folium.plugins import MarkerCluster # 지도 위 다수의 포인트를 클러스터링해 가독성을 높이는 MarkerCluster 플러그인 임포트
import osmnx as ox # OpenStreetMap 데이터 다운로드 및 네트워크 그래프 생성을 위한 OSMnx 임포트 (별칭: ox)
import networkx as nx # 그래프(노드-엣지 구조) 기반 최단경로 및 네트워크 분석을 위한 NetworkX 임포트 (별칭: nx)
from shapely.geometry import LineString # 두 점 이상을 연결하는 선(LineString) 객체 생성을 위한 Shapely 클래스 임포트
from shapely.ops import unary_union # 여러 개의 Geometry를 하나로 병합(Union)하기 위한 Shapely 연산 함수 임포트
from shapely.geometry import mapping # Shapely Geometry 객체를 GeoJSON 형식(dict 구조)으로 변환하기 위한 함수 임포트

In [ ]:
BASE_DIR = os.getcwd() # 현재 파이썬 스크립트가 실행되고 있는 작업 디렉토리(Working Directory) 경로를 가져옴
DATA_DIR = os.path.join(BASE_DIR, "data") # BASE_DIR 경로 하위에 "data" 폴더 경로를 생성 (운영체제에 맞는 경로 구분자 자동 처리)
OUT_DIR  = os.path.join(BASE_DIR, "outputs") # BASE_DIR 경로 하위에 "outputs" 폴더 경로를 생성 (결과 파일 저장용 디렉토리)

os.makedirs(OUT_DIR, exist_ok=True) # OUT_DIR 폴더가 존재하지 않으면 생성하고, 이미 존재하면 에러 없이 넘어가도록 설정

In [ ]:
ADMIN_VEC  = os.path.join(DATA_DIR, "BND_ADM_DONG_PG.shp") # 행정동 경계 Shapefile 경로 설정 (data 폴더 내 BND_ADM_DONG_PG.shp)

TOP2_IDS = ["11210540", "11210630"] # 분석 대상으로 선택할 행정동 코드 2개 지정 (ADM_CD 기준)

ADMIN_COL_ID = "ADM_CD" # 행정동 코드가 저장된 컬럼명 정의
ADMIN_COL_NM = "ADM_NM" # 행정동 이름이 저장된 컬럼명 정의

gdf_admin = gpd.read_file(ADMIN_VEC) # 행정동 Shapefile을 GeoDataFrame 형태로 불러오기

gdf_admin["region_id"] = gdf_admin[ADMIN_COL_ID].astype(str).str.strip() # ADM_CD를 문자열로 변환하고 공백 제거 후 region_id 표준 컬럼 생성
gdf_admin["region_nm"] = gdf_admin[ADMIN_COL_NM].astype(str).str.strip() # ADM_NM을 문자열로 변환하고 공백 제거 후 region_nm 표준 컬럼 생성

if gdf_admin.crs is None:
    raise ValueError("행정동 shp CRS 없음 (.prj 확인)") # 좌표계(CRS)가 정의되지 않았으면 오류 발생 (.prj 파일 존재 여부 확인 필요)

gdf_admin = gdf_admin.to_crs(5179) # 좌표계를 EPSG:5179(미터 기반 투영좌표계, 거리 분석용)으로 변환
gdf_admin_seoul = gdf_admin[gdf_admin["region_id"].str.startswith("11")].copy() # ADM_CD 앞 2자리가 '11'인 서울 행정동만 필터링
gdf_sel = gdf_admin_seoul[gdf_admin_seoul["region_id"].isin(TOP2_IDS)].copy() # 서울 행정동 중에서 TOP2_IDS에 해당하는 행정동만 선택

sel_union = unary_union(gdf_sel.geometry) # 선택된 2개 행정동의 폴리곤을 하나의 Geometry로 병합 (Union)

OUT_GPKG   = os.path.join(OUT_DIR, "BND_ADM_DONG_PG.gpkg") # 결과를 저장할 GeoPackage 파일 경로 설정
OUT_LAYER  = "admin_top2" # GeoPackage 내부에 저장될 레이어 이름 지정

if os.path.exists(OUT_GPKG):
    os.remove(OUT_GPKG) # 동일 파일이 이미 존재하면 삭제 (중복 저장 방지)

gdf_sel.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG") # 선택된 행정동 GeoDataFrame을 GeoPackage 형식으로 저장

In [ ]:
BUS_XLSX   = os.path.join(DATA_DIR, "서울시버스정류소위치정보(20260108).xlsx") # 서울시 버스정류소 엑셀 파일 경로 설정

BUS_LON_COL  = "X좌표" # 버스정류소 경도(Longitude)가 저장된 컬럼명 정의
BUS_LAT_COL  = "Y좌표" # 버스정류소 위도(Latitude)가 저장된 컬럼명 정의

bus_raw = pd.read_excel(BUS_XLSX) # 엑셀 파일을 Pandas DataFrame으로 불러오기
bus_raw[BUS_LON_COL] = pd.to_numeric(bus_raw[BUS_LON_COL], errors="coerce") # 경도 컬럼을 숫자형으로 변환 (변환 불가 값은 NaN 처리)
bus_raw[BUS_LAT_COL] = pd.to_numeric(bus_raw[BUS_LAT_COL], errors="coerce") # 위도 컬럼을 숫자형으로 변환 (변환 불가 값은 NaN 처리)
bus_raw = bus_raw.dropna(subset=[BUS_LON_COL, BUS_LAT_COL]) # 경도 또는 위도 값이 없는 행(NaN)을 제거하여 좌표 오류 방지

gdf_bus = gpd.GeoDataFrame(
    bus_raw,
    geometry=gpd.points_from_xy(bus_raw[BUS_LON_COL], bus_raw[BUS_LAT_COL]),
    crs=4326
).to_crs(5179) # 위경도(4326) 기준으로 Point Geometry 생성 후, 거리 분석을 위해 EPSG:5179(미터 단위 투영좌표계)로 변환

gdf_bus_sel = gpd.sjoin(
    gdf_bus[["geometry"]],
    gdf_sel[["region_id", "region_nm", "geometry"]],
    how="inner", predicate="within" # predicate="within" → 포인트가 폴리곤 내부에 있을 때만 포함
).drop(columns="index_right") # 선택된 행정동(gdf_sel) 내부에 위치한 버스정류소만 공간조인으로 추출

gdf_bus_sel["stop_type"] = "bus" # 정류장 유형을 구분하기 위한 컬럼 추가 (버스 정류소 표시용)

In [ ]:
SUBWAY_CSV = os.path.join(DATA_DIR, "서울교통공사_1_8호선 역사 좌표(위경도) 정보_20250814.csv") # 서울교통공사 지하철역 좌표 CSV 파일 경로 설정 

SUB_LON_COL = "경도" # 지하철역 경도(Longitude)가 저장된 컬럼명 정의
SUB_LAT_COL = "위도" # 지하철역 위도(Latitude)가 저장된 컬럼명 정의

try:
    sub_raw = pd.read_csv(SUBWAY_CSV, encoding="utf-8") # UTF-8 인코딩으로 CSV 파일 읽기 시도
except UnicodeDecodeError:
    sub_raw = pd.read_csv(SUBWAY_CSV, encoding="cp949") # UTF-8 실패 시, 국내 공공데이터에서 자주 쓰이는 CP949 인코딩으로 재시도

sub_raw[SUB_LON_COL] = pd.to_numeric(sub_raw[SUB_LON_COL], errors="coerce") # 경도 컬럼을 숫자형으로 변환 (변환 불가 값은 NaN 처리)
sub_raw[SUB_LAT_COL] = pd.to_numeric(sub_raw[SUB_LAT_COL], errors="coerce") # 위도 컬럼을 숫자형으로 변환 (변환 불가 값은 NaN 처리)
sub_raw = sub_raw.dropna(subset=[SUB_LON_COL, SUB_LAT_COL]) # 경도 또는 위도 값이 결측(NaN)인 행 제거 → 잘못된 좌표 제거

gdf_sub = gpd.GeoDataFrame(
    sub_raw,
    geometry=gpd.points_from_xy(sub_raw[SUB_LON_COL], sub_raw[SUB_LAT_COL]),
    crs=4326
).to_crs(5179) # 위경도(WGS84, EPSG:4326) 기준 Point Geometry 생성 후, 거리 분석을 위해 미터 단위 투영좌표계(EPSG:5179)로 변환

gdf_sub_sel = gpd.sjoin(
    gdf_sub[["geometry"]],
    gdf_sel[["region_id", "region_nm", "geometry"]],
    how="inner", predicate="within" # predicate="within" → 포인트가 폴리곤 내부에 있는 경우만 포함
).drop(columns="index_right") # 선택된 행정동(gdf_sel) 내부에 위치한 지하철역만 공간조인으로 필터링

gdf_sub_sel["stop_type"] = "subway" # 정류장 유형 구분을 위한 컬럼 추가 (지하철역 표시용)

In [ ]:
BUS_BUFFER_M = 300.0 # 버스정류소 기준 직선거리 300m를 커버 범위로 설정 (미터 단위)
SUB_BUFFER_M = 500.0 # 지하철역 기준 직선거리 500m를 커버 범위로 설정 (미터 단위)

bus_buf = unary_union(gdf_bus_sel.geometry.buffer(BUS_BUFFER_M)) if len(gdf_bus_sel) else None 
# 선택된 버스정류소 포인트에 300m 버퍼를 생성한 뒤 모두 병합(union) / 정류장이 없으면 None 처리

sub_buf = unary_union(gdf_sub_sel.geometry.buffer(SUB_BUFFER_M)) if len(gdf_sub_sel) else None
# 선택된 지하철역 포인트에 500m 버퍼를 생성한 뒤 모두 병합(union) / 지하철역이 없으면 None 처리

bufs = [b for b in [bus_buf, sub_buf] if b is not None] # None이 아닌 버퍼 객체만 리스트로 수집
cover_buf = unary_union(bufs) if bufs else None # 버스 + 지하철 버퍼를 하나의 커버리지 영역으로 병합 / 버퍼가 하나도 없으면 None 처리

straight_uncovered_rows = [] # 직선거리 기준 비커버 지역을 저장할 리스트 초기화

for _, dong in gdf_sel.iterrows():     # 선택된 각 행정동 폴리곤을 하나씩 반복 처리
    unc = dong.geometry if cover_buf is None else dong.geometry.difference(cover_buf) 
    # 커버 영역이 없으면 행정동 전체가 비커버 / 있으면 행정동 폴리곤에서 커버 영역을 차집합(difference)으로 제거
    
    if unc and not unc.is_empty: # 차집합 결과가 존재하고 비어있지 않은 경우만 저장
        
        straight_uncovered_rows.append({
            "region_id": dong["region_id"],
            "region_nm": dong["region_nm"],
            "geometry":  unc
        }) # 행정동 코드, 이름, 비커버 Geometry를 리스트에 추가

gdf_unc_straight = gpd.GeoDataFrame(straight_uncovered_rows, crs=5179) # 직선거리 기반 비커버 지역을 GeoDataFrame으로 생성 (EPSG:5179 유지)

In [ ]:
GRAPH_BUFFER_M = 1500.0 # 네트워크 그래프를 생성할 때 행정동 경계를 1500m 확장하여 도로 데이터를 확보하기 위한 버퍼 거리

poly_graph_ll = (
    gpd.GeoSeries([sel_union.buffer(GRAPH_BUFFER_M)], crs=5179) # 선택된 행정동 통합 폴리곤(sel_union)에 1500m 버퍼를 생성 (EPSG:5179, 미터 단위)
    .to_crs(4326) # OSMnx는 위경도 좌표계(EPSG:4326)를 요구하므로 좌표계를 변환
    .iloc[0] # GeoSeries에서 실제 Polygon 객체 하나만 추출
)

ox.settings.log_console = False # OSMnx 실행 시 콘솔 로그 출력 비활성화 (불필요한 다운로드 로그 제거)

G = ox.graph_from_polygon(poly_graph_ll, network_type="walk", simplify=True)
# 확장된 폴리곤 영역을 기준으로 OpenStreetMap 보행 네트워크 그래프 생성
# network_type="walk" → 보행 가능한 도로만 수집
# simplify=True → 교차점 기준으로 네트워크를 단순화하여 노드/엣지 정리

In [ ]:
gdf_stop = gpd.GeoDataFrame(
    pd.concat([gdf_bus_sel, gdf_sub_sel], ignore_index=True),
    geometry="geometry", crs=5179
) # 버스 + 지하철 정류장을 하나의 GeoDataFrame으로 통합 (EPSG:5179 유지)

gdf_stop_ll = gdf_stop.to_crs(4326).copy() # OSMnx 네트워크 매칭을 위해 위경도 좌표계(EPSG:4326)로 변환

gdf_stop_ll["v_node"] = ox.distance.nearest_nodes(
    G, 
    X=gdf_stop_ll.geometry.x.values, 
    Y=gdf_stop_ll.geometry.y.values
)
# 각 정류장 포인트에 대해 네트워크 그래프 G 상에서 가장 가까운 노드 ID를 계산
# X=경도, Y=위도 배열을 전달하여 벡터 방식으로 최근접 노드 매핑

In [ ]:
iso_polys_m = [] # 각 정류장별 네트워크 기반 커버 폴리곤(미터 단위)을 저장할 리스트
reachable_edge_rows = [] # 도달 가능한 도로 엣지(라인) 정보를 저장할 리스트

reachable_edge_seen = set() # 중복 엣지 저장을 방지하기 위한 방문 체크용 set

EDGE_BUFFER_M = 25.0   # 도로 폭 근사(20~50 권장) / 네트워크 라인을 폴리곤으로 만들기 위해 도로 폭을 근사한 버퍼 거리(미터)

for _, r in gdf_stop_ll.iterrows(): # 각 정류장(버스/지하철)에 대해 반복 수행
    v = int(r["v_node"]) # 해당 정류장이 매핑된 네트워크 노드 ID
    
    cutoff = BUS_BUFFER_M if r["stop_type"] == "bus" else SUB_BUFFER_M # 정류장 유형에 따라 네트워크 탐색 반경(도보 거리) 설정

    try:
        Gsub = nx.ego_graph(G, v, radius=cutoff, distance="length", undirected=True)
        # 기준 노드 v에서 cutoff 거리 이내의 서브그래프 생성
        # distance="length" → 엣지 길이(미터) 기준으로 탐색
    except Exception:
        continue
        # 네트워크 탐색 실패 시 해당 정류장은 건너뜀

    if Gsub.number_of_edges() == 0:
        continue
        # 도달 가능한 엣지가 없으면 다음 정류장으로 이동

    _, gdf_edges_ll = ox.graph_to_gdfs(Gsub, nodes=True, edges=True, fill_edge_geometry=True)
    # 서브그래프를 GeoDataFrame으로 변환 (엣지 Geometry 포함)

    for idx, ed in gdf_edges_ll.iterrows(): # 서브그래프 내 모든 엣지를 반복
        u, w, k  = idx  # 멀티인덱스 (u, v, key) / 네트워크 엣지는 (출발노드, 도착노드, key) 구조
        edge_key = (min(u, w), max(u, w), k) # 방향성 제거를 위해 노드번호 정렬 후 고유 키 생성
        
        if edge_key in reachable_edge_seen:
            continue
            # 이미 처리된 엣지는 중복 저장 방지
        
        reachable_edge_seen.add(edge_key) # 신규 엣지를 방문 집합에 기록
        geom = ed.geometry # 해당 엣지의 LineString Geometry
        
        length_m = float(ed.get("length", 0.0)) # 엣지 길이(미터) 추출 (없으면 0 처리)
        
        if geom and not geom.is_empty and length_m > 0:
            reachable_edge_rows.append({"length_m": length_m, "geometry": geom})
            # 유효한 엣지만 리스트에 저장

    poly_m = unary_union(gdf_edges_ll.to_crs(5179).geometry.buffer(EDGE_BUFFER_M))
    # 서브그래프 엣지를 5179(미터) 좌표계로 변환 후 도로폭만큼 버퍼 생성/  여러 엣지를 하나의 폴리곤으로 병합(union)

    if poly_m and not poly_m.is_empty:
        iso_polys_m.append(poly_m) # 정류장별 네트워크 커버 폴리곤을 리스트에 저장

cover_iso_m = unary_union(iso_polys_m) if iso_polys_m else None # 모든 정류장의 커버 폴리곤을 하나로 병합하여 최종 Isochrone 생성

if not cover_iso_m or cover_iso_m.is_empty:
    raise ValueError("Isochrone 커버 폴리곤이 비어있음 (stop/네트워크 범위 확인)") # 커버 폴리곤이 생성되지 않으면 네트워크 범위 또는 정류장 매칭 오류

gdf_edges_reachable = gpd.GeoDataFrame(reachable_edge_rows, geometry="geometry", crs=4326) # 도달 가능한 모든 네트워크 엣지를 GeoDataFrame으로 생성 (위경도 좌표계)
gdf_cover_iso = gpd.GeoDataFrame([{"geometry": cover_iso_m}], crs=5179) # 최종 네트워크 기반 커버 폴리곤을 GeoDataFrame으로 생성 (미터 좌표계)

In [ ]:
uncovered_rows = [] # 네트워크 기반 비커버 지역을 저장할 리스트 초기화

for _, dong in gdf_sel.iterrows(): # 선택된 각 행정동 폴리곤에 대해 반복 처리
    unc = dong.geometry.difference(cover_iso_m) # 행정동 폴리곤에서 네트워크 커버 영역(isochrone)을 차집합으로 제거
    if unc and not unc.is_empty: # 차집합 결과가 존재하고 비어있지 않은 경우만 저장
        uncovered_rows.append({
            "region_id": str(dong["region_id"]), # 행정동 코드 (문자열 변환하여 일관성 유지)
            "region_nm": dong["region_nm"], # 행정동 이름
            "geometry":  unc # 네트워크 기반 비커버 Geometry
        })

gdf_unc_network = gpd.GeoDataFrame(uncovered_rows, crs=5179) # 네트워크 기반 비커버 지역을 GeoDataFrame으로 생성 (미터 좌표계 유지)

gdf_sel2 = gdf_sel.copy() # 원본 행정동 GeoDataFrame을 복사하여 지표 계산용 데이터 생성
gdf_sel2["admin_area_m2"] = gdf_sel2.geometry.area # 각 행정동의 전체 면적 계산 (EPSG:5179 기준 → 제곱미터 단위)

if len(gdf_unc_network) > 0: # 비커버 지역이 존재하는 경우
    unc_area = gdf_unc_network.copy() # 비커버 GeoDataFrame 복사
    unc_area["uncovered_area_m2"] = unc_area.geometry.area # 비커버 지역 면적 계산 (제곱미터 단위)
    
    gdf_sel2 = gdf_sel2.merge(
        unc_area[["region_id", "uncovered_area_m2"]],
        on="region_id",
        how="left"
    ) # 행정동 코드 기준으로 비커버 면적을 병합
    
    gdf_sel2["uncovered_area_m2"] = gdf_sel2["uncovered_area_m2"].fillna(0.0) # 비커버 지역이 없는 행정동은 0으로 처리
else:
    gdf_sel2["uncovered_area_m2"] = 0.0 # 비커버 지역이 전혀 없는 경우 모든 값을 0으로 설정

gdf_sel2["uncovered_ratio"] = np.where(
    gdf_sel2["admin_area_m2"] > 0,
    gdf_sel2["uncovered_area_m2"] / gdf_sel2["admin_area_m2"],
    0.0
) # 행정동 전체 면적 대비 비커버 면적 비율 계산 / 면적이 0인 경우를 대비해 조건 처리

In [ ]:
GRID_SHP   = os.path.join(DATA_DIR, "nlsp_021001001.shp") # 100m 인구격자 Shapefile 경로 설정

GRID_ID_COL  = "gid" # 격자 고유 ID 컬럼명 정의
GRID_POP_COL = "val" # 인구 값이 저장된 원본 컬럼명 정의

gdf_grid = gpd.read_file(GRID_SHP) # 인구격자 Shapefile을 GeoDataFrame으로 불러오기

gdf_grid = gdf_grid.to_crs(5179) # 거리 및 면적 계산을 위해 EPSG:5179(미터 단위 투영좌표계)로 변환

gdf_grid[GRID_ID_COL] = gdf_grid[GRID_ID_COL].astype(str).str.strip() # 격자 ID를 문자열로 변환하고 공백 제거 (조인 안정성 확보)
gdf_grid["pop"] = pd.to_numeric(gdf_grid[GRID_POP_COL], errors="coerce").fillna(0.0) # 인구 값을 숫자형으로 변환 (변환 불가 값은 0으로 처리)

gdf_grid_cut = gdf_grid[gdf_grid.geometry.intersects(sel_union.buffer(0))].copy()
# 선택된 행정동 영역과 교차(intersects)하는 격자만 필터링 / buffer(0)는 Geometry 오류 방지용 정합 처리

gdf_grid_cut = gpd.sjoin(
    gdf_grid_cut[[GRID_ID_COL, "pop", "geometry"]],
    gdf_sel[["region_id", "region_nm", "geometry"]],
    how="inner", predicate="intersects"
).drop(columns="index_right") # 격자를 행정동과 공간조인하여 각 격자에 행정동 코드/이름 매핑

gdf_grid_cut["pop"] = gdf_grid_cut["pop"].fillna(0.0) # 혹시 남아 있을 수 있는 인구 결측값을 0으로 처리

In [ ]:
USE_RATIO_FILTER = True # 격자의 절반 이상이 비커버 지역에 속하는 경우만 인정할지 여부 설정

priority_rows = [] # 최종 우선순위 격자 정보를 저장할 리스트 초기화

for rid, g_dong in gdf_grid_cut.groupby("region_id"): # 행정동(region_id)별로 인구격자 그룹화하여 반복
    unc_rows = gdf_unc_network[gdf_unc_network["region_id"].astype(str) == str(rid)] # 해당 행정동의 네트워크 기반 비커버 폴리곤 추출
    
    if len(unc_rows) == 0:
        continue # 비커버 지역이 없으면 해당 행정동은 건너뜀

    unc_poly = unary_union(unc_rows.geometry) # 해당 행정동의 비커버 영역을 하나의 폴리곤으로 병합
    g_unc = g_dong[g_dong.geometry.intersects(unc_poly)].copy() # 비커버 영역과 교차하는 인구격자만 추출
    
    if len(g_unc) == 0:
        continue # 비커버 지역과 겹치는 격자가 없으면 건너뜀

    if USE_RATIO_FILTER: # 격자의 일정 비율 이상이 비커버 지역에 포함되는지 필터링 수행
        g_unc["cell_area_m2"] = g_unc.geometry.area # 각 격자의 전체 면적 계산
        g_unc["unc_area_m2"] = g_unc.geometry.intersection(unc_poly).area # 격자와 비커버 영역의 교차 면적 계산
        g_unc["unc_ratio"] = np.where(
            g_unc["cell_area_m2"] > 0,
            g_unc["unc_area_m2"] / g_unc["cell_area_m2"],
            0.0
        ) # 격자 전체 면적 대비 비커버 면적 비율 계산
        
        g_unc = g_unc[g_unc["unc_ratio"] >= 0.5].copy() # 비커버 비율이 50% 이상인 격자만 유지
        
        if len(g_unc) == 0:
            continue # 조건을 만족하는 격자가 없으면 건너뜀

    g_unc = g_unc.sort_values("pop", ascending=False).head(999).copy() # 인구(pop) 기준 내림차순 정렬 후 상위 999개 격자 선택
    g_unc["rank"] = np.arange(1, len(g_unc) + 1) # 인구 기준 우선순위(rank) 부여 (1부터 시작)

    for _, rr in g_unc.iterrows(): # 우선순위 격자를 하나씩 리스트에 저장
        
        priority_rows.append({
            "region_id": str(rr.get("region_id", "")), # 행정동 코드
            "region_nm": rr.get("region_nm", ""), # 행정동 이름
            "gid": rr.get(GRID_ID_COL, ""), # 격자 고유 ID
            "pop": float(rr.get("pop", 0)), # 격자 인구
            "rank": int(rr.get("rank", 0)), # 인구 기준 순위      
            "geometry": rr.geometry # 격자 Geometry
        })

gdf_priority = gpd.GeoDataFrame(priority_rows, crs=5179) # 최종 우선순위 격자 목록을 GeoDataFrame으로 생성 (미터 좌표계 유지)

In [ ]:
gdf_stop_m = gdf_stop_ll.to_crs(5179).copy() # 정류장 포인트를 거리 계산이 쉬운 미터 좌표계(EPSG:5179)로 변환해 복사

route_lines = [] # 우선순위 격자 → 최근접 정류장까지의 최단경로(LineString) 결과를 저장할 리스트 초기화

if len(gdf_priority) > 0 and len(gdf_stop_ll) > 0: # 우선순위 격자와 정류장 데이터가 모두 있을 때만 경로 계산 수행
    gdf_pri_ll = gdf_priority.to_crs(4326).copy() # 우선순위 격자를 OSMnx/NetworkX용 위경도(EPSG:4326)로 변환해 복사
    pri_cent_ll = gdf_pri_ll.geometry.centroid # 각 우선순위 격자의 중심점(centroid)을 계산 (위경도 기준)

    gdf_pri_ll["src_node"] = ox.distance.nearest_nodes(
        G, X=pri_cent_ll.x.values, Y=pri_cent_ll.y.values
    ) # 각 격자 중심점에 대해 네트워크 그래프 G 상의 최근접 노드(src_node)를 매핑

    for _, rr in gdf_pri_ll.iterrows(): # 우선순위 격자(각 행)를 하나씩 순회하며 경로를 계산
        c_m = gpd.GeoSeries([rr.geometry.centroid], crs=4326).to_crs(5179).iloc[0] # 해당 격자 중심점을 미터 좌표계(5179)로 변환해 유클리드 거리 계산에 사용
        gdf_stop_m["d_eucl"] = gdf_stop_m.geometry.distance(c_m) # 격자 중심점과 모든 정류장 사이의 직선거리(유클리드 거리)를 계산
        cand = gdf_stop_m.nsmallest(20, "d_eucl")[["v_node", "stop_type"]] # 직선거리 기준 가장 가까운 정류장 후보 20개만 추려 네트워크 계산 비용을 줄임
        src = int(rr["src_node"]) # 출발 노드(격자 중심에 매핑된 네트워크 노드)를 정수형으로 확보
        best_path, best_len, best_type = None, np.inf, None # 최적(최단) 경로, 최단거리, 도착 정류장 타입을 저장할 변수 초기화

        for _, cs in cand.iterrows(): # 후보 정류장 20개에 대해 실제 네트워크 최단거리로 최적 정류장을 탐색

            tgt = int(cs["v_node"]) # 후보 정류장의 네트워크 노드 ID를 도착 노드로 설정

            try:
                length = nx.shortest_path_length(G, src, tgt, weight="length") # 출발→도착 노드 간 네트워크 최단거리(엣지 length 가중치)를 계산
                if length < best_len:
                    best_len  = length # 더 짧은 경로가 나오면 최단거리 갱신
                    best_path = nx.shortest_path(G, src, tgt, weight="length") # 최단경로의 노드 시퀀스를 저장 (실제 라인 생성용)
                    best_type = cs["stop_type"] # 최적 도착 정류장의 타입(bus/subway)을 저장
            except Exception: 
                continue # 경로 계산 실패(단절/오류) 시 해당 후보는 건너뜀

        if best_path is None or not np.isfinite(best_len):
            continue # 최적 경로를 찾지 못했거나 거리값이 비정상이면 해당 격자는 제외

        coords = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in best_path] # 최단경로의 각 노드에서 (경도 x, 위도 y) 좌표를 추출해 라인 좌표열 생성

        if len(coords) < 2:
            continue # 라인을 만들 최소 2개 좌표가 없으면 제외

        route_lines.append({
            "region_id":    rr.get("region_id", ""), # 해당 격자가 속한 행정동 코드 저장
            "region_nm":    rr.get("region_nm", ""), # 해당 격자가 속한 행정동 이름 저장
            "gid":          rr.get("gid", ""), # 격자 고유 ID 저장
            "pop":          float(rr.get("pop", 0)), # 격자 인구(pop) 저장
            "rank":         int(rr.get("rank", 0)),  # 우선순위 순위(rank) 저장
            "dist_m":       float(best_len), # 네트워크 최단거리(미터)를 저장
            "to_stop_type": str(best_type), # 최단거리로 연결된 도착 정류장 유형(bus/subway) 저장
            "geometry":     LineString(coords) # 최단경로 좌표열로 LineString을 생성하여 경로 Geometry로 저장
        })

gdf_routes_ll = gpd.GeoDataFrame(route_lines, crs=4326) # 생성된 모든 최단경로 라인을 GeoDataFrame으로 만들고 좌표계는 위경도(4326)로 지정

In [ ]:
admin2_ll = gdf_sel2.to_crs(4326) # 행정동 + 비커버 지표가 포함된 GeoDataFrame을 Folium용 위경도(EPSG:4326)로 변환
admin_ll = gdf_sel.to_crs(4326) # 원본 선택 행정동 폴리곤을 위경도 좌표계로 변환 (지도 경계 표시용)
bus_ll = gdf_bus_sel.to_crs(4326) # 선택된 버스정류소 포인트를 위경도 좌표계로 변환 (지도 시각화용)
sub_ll = gdf_sub_sel.to_crs(4326) # 선택된 지하철역 포인트를 위경도 좌표계로 변환 (지도 시각화용)
unc_straight_ll = gdf_unc_straight.to_crs(4326) if len(gdf_unc_straight) else gpd.GeoDataFrame([], crs=4326)  # 직선거리 기반 비커버 지역이 존재하면 위경도로 변환, 없으면 빈 GeoDataFrame 생성
unc_network_ll = gdf_unc_network.to_crs(4326) if len(gdf_unc_network) else gpd.GeoDataFrame([], crs=4326) # 네트워크 기반 비커버 지역이 존재하면 위경도로 변환, 없으면 빈 GeoDataFrame 생성
cover_iso_ll = gdf_cover_iso.to_crs(4326) # 네트워크 기반 Isochrone 커버 폴리곤을 위경도로 변환
grid_priority_ll = gdf_priority.to_crs(4326) if len(gdf_priority) else gpd.GeoDataFrame([], crs=4326) # 우선순위 인구격자가 존재하면 위경도로 변환, 없으면 빈 GeoDataFrame 생성

bounds = admin_ll.total_bounds # 선택 행정동 전체의 최소/최대 경도·위도 범위를 계산 (minx, miny, maxx, maxy)
center_lat = (bounds[1] + bounds[3]) / 2 # 지도 중심 위도 계산 (miny와 maxy의 평균)
center_lon = (bounds[0] + bounds[2]) / 2 # 지도 중심 경도 계산 (minx와 maxx의 평균)

In [ ]:
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")
# 지도 중심(center_lat/lon)과 초기 줌(13), CartoDB Positron 타일로 Folium 지도 객체 생성

# (1) 행정동(Top2)
folium.GeoJson(
    admin2_ll,
    # 비커버비율(uncovered_ratio)이 포함된 Top2 행정동 폴리곤 레이어를 GeoJson으로 추가
    name="행정동(Top2)",
    # 레이어 컨트롤에 표시될 이름 지정
    style_function=lambda x: {"fillOpacity": 0.05, "color": "#444444", "weight": 3},
    # 행정동 폴리곤 스타일(채움 매우 연하게, 테두리 진회색, 두께 3)
    tooltip=folium.GeoJsonTooltip(
        fields=["region_nm", "region_id", "uncovered_ratio"],
        # 툴팁에 표시할 속성 필드 지정
        aliases=["행정동", "코드", "비커버비율(참고)"],
        # 툴팁 라벨을 한글로 매핑
        localize=True
        # 숫자 표시 등을 로케일 형식으로 출력
    )
).add_to(m)
# 행정동 GeoJson 레이어를 지도(m)에 추가

# (2) 비커버(직선 버퍼)
if len(unc_straight_ll) > 0:
# 직선거리(버퍼) 기반 비커버 폴리곤이 있을 때만 레이어를 추가

    folium.GeoJson(
        unc_straight_ll,
        # 직선거리 버퍼 기준 비커버 폴리곤 레이어를 GeoJson으로 추가
        name="비커버(직선 버퍼: bus300/sub500)",
        # 레이어 컨트롤에 표시될 이름 지정
        style_function=lambda x: {"fillOpacity": 0.30, "color": "#cc0000", "weight": 2},
        # 비커버 폴리곤 스타일(빨간 테두리, 중간 채움 투명도)
        tooltip=folium.GeoJsonTooltip(fields=["region_nm"], aliases=["행정동"])
        # 툴팁에 행정동명만 표시
    ).add_to(m)
    # 직선 비커버 레이어를 지도(m)에 추가

# (3) 네트워크 커버(Isochrone)
folium.GeoJson(
    cover_iso_ll,
    # 네트워크 기반 커버 영역(Isochrone 폴리곤) 레이어를 GeoJson으로 추가
    name="네트워크 커버(Isochrone 폴리곤)",
    # 레이어 컨트롤에 표시될 이름 지정
    style_function=lambda x: {"fillOpacity": 0.12, "color": "#0066ff", "weight": 2}
    # 커버 폴리곤 스타일(파란 테두리, 낮은 채움 투명도)
).add_to(m)
# 네트워크 커버(Isochrone) 레이어를 지도(m)에 추가

# (4) 비커버(네트워크)
if len(unc_network_ll) > 0:
# 네트워크 기반 비커버 폴리곤이 있을 때만 레이어를 추가

    folium.GeoJson(
        unc_network_ll,
        # 행정동에서 Isochrone 커버를 뺀 네트워크 기반 비커버 폴리곤 레이어를 GeoJson으로 추가
        name="비커버(네트워크: 행정동-isochrone)",
        # 레이어 컨트롤에 표시될 이름 지정
        style_function=lambda x: {"fillOpacity": 0.25, "color": "#7a00cc", "weight": 2},
        # 네트워크 비커버 폴리곤 스타일(보라 테두리, 중간 채움 투명도)
        tooltip=folium.GeoJsonTooltip(fields=["region_nm"], aliases=["행정동"])
        # 툴팁에 행정동명 표시
    ).add_to(m)
    # 네트워크 비커버 레이어를 지도(m)에 추가

# (5) 버스정류장 — 초록 둥근 사각형 아이콘 (B)
fg_bus = folium.FeatureGroup(name="버스정류장", show=False)
# 버스정류장 마커를 묶어서 토글하기 위한 FeatureGroup 생성(기본 숨김)

for _, r in bus_ll.iterrows():
# 버스정류장 포인트를 한 개씩 순회하며 마커 추가

    folium.Marker(
        location=[r.geometry.y, r.geometry.x],
        # Folium 마커 위치는 [위도, 경도] 순서로 지정
        tooltip=f"🚌 버스 | {r.get('region_nm', '')}",
        # 마우스오버 시 버스 아이콘과 행정동명을 표시
        icon=folium.DivIcon(
            html="""
                <div style="
                    display:flex; align-items:center; justify-content:center;
                    width:20px; height:20px; font-size:11px;
                    background:#1db954; color:#fff;
                    border:2px solid #fff; border-radius:5px;
                    box-shadow:0 2px 5px rgba(0,0,0,0.35);
                    font-weight:900;
                ">B</div>
            """,
            # 초록색 둥근 사각형 배지 형태의 HTML 아이콘(B) 정의
            icon_size=(20, 20),
            # 아이콘의 표시 크기 지정
            icon_anchor=(10, 10)
            # 아이콘의 기준점(중심) 설정
        )
    ).add_to(fg_bus)
    # 버스정류장 마커를 버스 FeatureGroup에 추가

fg_bus.add_to(m)
# 버스정류장 FeatureGroup을 지도(m)에 추가

# (6) 지하철역 — 남색 원형 아이콘 (M)
fg_sub = folium.FeatureGroup(name="지하철역", show=False)
# 지하철역 마커를 묶어서 토글하기 위한 FeatureGroup 생성(기본 숨김)

for _, r in sub_ll.iterrows():
# 지하철역 포인트를 한 개씩 순회하며 마커 추가

    folium.Marker(
        location=[r.geometry.y, r.geometry.x],
        # Folium 마커 위치는 [위도, 경도] 순서로 지정
        tooltip=f"🚇 지하철 | {r.get('region_nm', '')}",
        # 마우스오버 시 지하철 아이콘과 행정동명을 표시
        icon=folium.DivIcon(
            html="""
                <div style="
                    display:flex; align-items:center; justify-content:center;
                    width:20px; height:20px; font-size:11px;
                    background:#1a3cc8; color:#fff;
                    border:2px solid #fff; border-radius:50%;
                    box-shadow:0 2px 5px rgba(0,0,0,0.35);
                    font-weight:900;
                ">M</div>
            """,
            # 남색 원형 배지 형태의 HTML 아이콘(M) 정의
            icon_size=(20, 20),
            # 아이콘의 표시 크기 지정
            icon_anchor=(10, 10)
            # 아이콘의 기준점(중심) 설정
        )
    ).add_to(fg_sub)
    # 지하철역 마커를 지하철 FeatureGroup에 추가

fg_sub.add_to(m)
# 지하철역 FeatureGroup을 지도(m)에 추가

# (7) 도달가능 도로망
fg_edges = folium.FeatureGroup(name="도달가능 도로망(네트워크 cutoff)", show=True)
# 네트워크 cutoff로 도달 가능한 도로망을 토글할 FeatureGroup 생성(기본 표시)

for _, rr in gdf_edges_reachable.iterrows():
# 도달 가능한 엣지(LineString/MultiLineString)를 순회하며 지도에 표시

    geom = rr.geometry
    # 해당 엣지의 Geometry를 가져옴

    if geom is None or geom.is_empty:
        continue
        # Geometry가 없거나 비어 있으면 건너뜀

    lines = [geom] if geom.geom_type == "LineString" else list(geom.geoms)
    # LineString은 그대로 사용, MultiLineString은 구성 요소 라인으로 분해

    for ls in lines:
        folium.PolyLine([(y, x) for x, y in ls.coords], weight=2, opacity=0.35).add_to(fg_edges)
        # (경도,위도)를 Folium용 (위도,경도)로 바꿔 PolyLine으로 추가

fg_edges.add_to(m)
# 도달가능 도로망 FeatureGroup을 지도(m)에 추가

# (8) 우선순위 격자 — 행정동별 rank 1만 표시
top1_ll = grid_priority_ll[grid_priority_ll["rank"] == 1].copy()  # 행정동별 최대 인구 격자 1개씩
# 우선순위 격자 중 각 행정동에서 인구가 가장 큰 격자(rank=1)만 추출

fg_pri = folium.FeatureGroup(name="우선순위 격자 TOP1 (행정동별 비커버 최대 인구)", show=True)
# 우선순위 TOP1 격자를 토글할 FeatureGroup 생성(기본 표시)

for _, rr in top1_ll.iterrows():
# TOP1 격자를 순회하며 폴리곤과 라벨 마커를 지도에 추가

    tip = f"{rr.get('region_nm','')} | rank {rr.get('rank','')} | pop {rr.get('pop',0):,.0f} | gid {rr.get('gid','')}"
    # 툴팁 문자열(행정동/순위/인구/격자ID)을 구성

    folium.GeoJson(
        {"type": "Feature", "properties": {}, "geometry": mapping(rr.geometry)},
        # 격자 폴리곤을 GeoJSON Feature로 변환해 GeoJson 레이어로 추가
        style_function=lambda x: {"fillOpacity": 0.65, "color": "#ff6600", "weight": 3},
        # 격자 폴리곤 스타일(주황 테두리, 높은 채움 불투명도)
        tooltip=tip
        # 격자 폴리곤에 툴팁을 연결
    ).add_to(fg_pri)
    # 우선순위 격자 폴리곤을 FeatureGroup에 추가

    c = rr.geometry.centroid
    # 격자 폴리곤의 중심점을 계산해 라벨 마커 위치로 사용

    folium.Marker(
        location=[c.y, c.x],
        # 중심점 좌표에 라벨 마커를 배치([위도,경도])
        tooltip=tip,
        # 라벨 마커에도 동일한 툴팁을 연결
        icon=folium.DivIcon(html=f"""
            <div style="
                display:flex; align-items:center; justify-content:center;
                width:34px; height:34px;
                font-size:15px; font-weight:900; color:#fff;
                background:#ff6600; border:2.5px solid #fff;
                border-radius:50%; box-shadow:0 2px 8px rgba(0,0,0,0.5);
            ">1</div>
        """)
        # 숫자 1이 표시되는 원형 배지 형태의 커스텀 HTML 아이콘을 사용
    ).add_to(fg_pri)
    # 라벨 마커를 우선순위 FeatureGroup에 추가

fg_pri.add_to(m)
# 우선순위 격자 FeatureGroup을 지도(m)에 추가

# ── 저장 ──────────────────────────────────────────────────────────────────────
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
# 행정동 바운딩박스에 맞게 지도를 자동으로 확대/이동하여 전체가 보이도록 맞춤

folium.LayerControl(collapsed=False).add_to(m)
# 레이어 토글 컨트롤을 지도에 추가(접지 않고 펼친 상태로 표시)

OUT_HTML = os.path.join(OUT_DIR, "lecture3_isochrone_uncovered_priority_pop.html")
# 결과 지도를 저장할 HTML 파일 경로 생성

m.save(OUT_HTML)
# Folium 지도를 HTML 파일로 저장

print("HTML 저장 완료:", OUT_HTML)
# 저장 완료 메시지와 저장 경로 출력

print(f"전체 격자 수: {len(gdf_priority)} → 표시: {len(top1_ll)} (행정동별 TOP1)")
# 전체 우선순위 후보 격자 수와 실제 표시된 TOP1 격자 수를 요약 출력

print(gdf_sel2[["region_nm", "region_id", "uncovered_ratio"]].sort_values("uncovered_ratio", ascending=False))
# 행정동별 비커버 비율을 내림차순으로 정렬해 출력(참고용)

print("\n행정동별 TOP1 격자:")
# 다음 출력 블록(행정동별 TOP1 격자 목록)의 제목을 출력

print(top1_ll[["region_nm", "gid", "pop", "rank"]].to_string(index=False))
# 행정동별 TOP1 격자의 핵심 컬럼(행정동/격자ID/인구/순위)을 표 형태 문자열로 출력

In [46]:
# =========================================================
# [부록] pydeck 인구격자 시각화 — 데이터 준비
# =========================================================
# %pip install pydeck   # 설치가 안 돼 있으면 주석 해제 후 실행

import pydeck as pdk

# 격자 중심점(centroid)을 4326(위경도)으로 변환하여 pydeck 입력 포인트 생성
gdf_grid_pts = gdf_grid_cut[["gid", "pop", "geometry"]].copy()
gdf_grid_pts = gdf_grid_pts.to_crs(4326)                          # 4326 변환
gdf_grid_pts["lon"] = gdf_grid_pts.geometry.centroid.x            # 경도
gdf_grid_pts["lat"] = gdf_grid_pts.geometry.centroid.y            # 위도
gdf_grid_pts["pop"] = gdf_grid_pts["pop"].clip(lower=0)           # 음수 제거

df_pts = gdf_grid_pts[["lon", "lat", "pop"]].dropna().copy()      # pydeck용 DataFrame
df_pts["pop"] = df_pts["pop"].astype(float)

포인트 수: 430
              lon         lat          pop
count  430.000000  430.000000   430.000000
mean   126.975709   37.466701    84.395349
std      0.007659    0.010091   187.302299
min    126.956449   37.449451     0.000000
25%    126.971422   37.459367     0.000000
50%    126.975934   37.464826     0.000000
75%    126.981489   37.471998     0.000000
max    126.988355   37.493575  1262.000000


In [ ]:
# =========================================================
# [부록 3] HeatmapLayer — 격자 인구 밀도를 열지도로 표현
# =========================================================
# 참고: https://deckgl.readthedocs.io/en/latest/gallery/heatmap_layer.html
#
# - 포인트를 커널 밀도 추정(KDE)으로 부드럽게 블렌딩
# - 3D 돌출 없이 2D 색상 그라데이션으로 밀도 표현
# - get_weight: 인구수를 가중치로 사용 → 인구 밀집 지역 강조
# - radiusPixels: 각 포인트가 퍼지는 픽셀 반경 (zoom에 따라 조절)
# - intensity / threshold: 전체 밝기 및 가장자리 투명도 제어

# pop 가중치 정규화 (0~1 스케일) — HeatmapLayer 안정성 향상
pop_max = df_pts["pop"].max()
df_heat = df_pts.copy()
df_heat["weight"] = (df_heat["pop"] / pop_max).clip(lower=0.0)  # 0~1 정규화

heatmap_layer = pdk.Layer(
    "HeatmapLayer",
    data=df_heat,                              # 포인트 DataFrame
    get_position=["lon", "lat"],               # 위치 컬럼
    get_weight="weight",                       # 가중치: 정규화된 인구
    radius_pixels=60,                          # 커널 반경 (픽셀) — zoom 13 기준 약 100~150m
    intensity=1.5,                             # 전체 밝기 배율
    threshold=0.03,                            # 이 값 미만 픽셀은 투명 처리 (가장자리 정리)
    color_range=[                              # 저밀도(파랑) → 고밀도(빨강) 5단계
        [0,   0,   255, 0  ],                  # 완전 투명 (배경)
        [0,   128, 255, 120],                  # 연파랑
        [0,   255, 128, 180],                  # 청록
        [255, 200, 0,   220],                  # 노랑
        [255, 80,  0,   240],                  # 주황
        [200, 0,   0,   255],                  # 진빨강 (고밀도)
    ],
    aggregation="SUM",                         # 겹치는 포인트 합산
    pickable=False,                            # HeatmapLayer는 픽셀 단위라 tooltip 미지원
)

view_heat = pdk.ViewState(
    latitude=float(center_ll_pts["lat"]),
    longitude=float(center_ll_pts["lon"]),
    zoom=13,
    pitch=0,                                   # 2D 평면 뷰 (히트맵은 부감이 더 명확)
    bearing=0,
)

deck_heat = pdk.Deck(
    layers=[heatmap_layer],
    initial_view_state=view_heat,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",  # 어두운 배경 → 열지도 대비 극대화
    tooltip=False,
)

# ── Jupyter 인라인 렌더링 ──────────────────────────────────
deck_heat

# ── HTML로 저장하려면 아래 주석 해제 ─────────────────────
# deck_heat.to_html(os.path.join(OUT_DIR, "pydeck_heatmap.html"), open_browser=False)
# print("저장:", os.path.join(OUT_DIR, "pydeck_heatmap.html"))

In [47]:
# =========================================================
# [부록 1] HexagonLayer — 격자 인구를 육각형 기둥으로 집계
# =========================================================
# 참고: https://deckgl.readthedocs.io/en/latest/gallery/hexagon_layer.html
#
# - radius(m) 단위로 포인트를 자동 집계 → 높이/색상으로 밀도 표현
# - get_position: [경도, 위도] 포인트 컬럼
# - elevation_scale: 높이 배율 (값이 클수록 기둥이 높아짐)
# - colorRange: 저밀도(차가운 색) → 고밀도(따뜻한 색)

# 지도 중심점
center_ll_pts = df_pts[["lat", "lon"]].mean()

hexagon_layer = pdk.Layer(
    "HexagonLayer",
    data=df_pts,                               # 포인트 DataFrame
    get_position=["lon", "lat"],               # 위치 컬럼
    radius=150,                                # 육각형 반경 (m) — 격자 100m보다 약간 크게
    elevation_scale=8,                         # 높이 배율
    elevation_range=[0, 800],                  # 최소/최대 높이 (m 단위 화면 값)
    extruded=True,                             # 3D 돌출 ON
    pickable=True,                             # 클릭 상호작용
    auto_highlight=True,                       # 마우스오버 강조
    color_range=[                              # 6단계 색상 (파랑 → 빨강)
        [1,  152, 189, 200],
        [73, 227, 206, 200],
        [216,254, 181, 200],
        [254,237, 177, 200],
        [254,173,  84, 200],
        [209,  55,  78, 200],
    ],
    get_elevation_weight="pop",                # 높이 집계 기준: 인구
    elevation_aggregation="SUM",               # 집계 방식: 합계
    get_color_weight="pop",                    # 색상 집계 기준: 인구
    color_aggregation="SUM",                   # 집계 방식: 합계
)

view_hex = pdk.ViewState(
    latitude=float(center_ll_pts["lat"]),
    longitude=float(center_ll_pts["lon"]),
    zoom=13,
    pitch=50,
    bearing=0,
)

deck_hex = pdk.Deck(
    layers=[hexagon_layer],
    initial_view_state=view_hex,
    map_style="https://basemaps.cartocdn.com/gl/positron-gl-style/style.json",
    tooltip={"text": "집계 인구: {elevationValue}\n격자 수: {colorValue}"},
)

# ── Jupyter 인라인 렌더링 ──────────────────────────────────
deck_hex                                                           # 셀 마지막 줄에 두면 인라인 표시

# ── HTML로 저장하려면 아래 주석 해제 ─────────────────────
# deck_hex.to_html(os.path.join(OUT_DIR, "pydeck_hexagon.html"), open_browser=False)
# print("저장:", os.path.join(OUT_DIR, "pydeck_hexagon.html"))

{
  "initialViewState": {
    "bearing": 0,
    "latitude": 37.46670139512002,
    "longitude": 126.9757092765704,
    "pitch": 50,
    "zoom": 13
  },
  "layers": [
    {
      "@@type": "HexagonLayer",
      "autoHighlight": true,
      "colorAggregation": "@@=SUM",
      "colorRange": [
        [
          1,
          152,
          189,
          200
        ],
        [
          73,
          227,
          206,
          200
        ],
        [
          216,
          254,
          181,
          200
        ],
        [
          254,
          237,
          177,
          200
        ],
        [
          254,
          173,
          84,
          200
        ],
        [
          209,
          55,
          78,
          200
        ]
      ],
      "data": [
        {
          "lat": 37.48634410183875,
          "lon": 126.95648802438059,
          "pop": 63.0
        },
        {
          "lat": 37.489949413120776,
          "lon": 126.95646190226587,
          "pop": 864.0
        },
        {
          "lat": 37.49085074059423,
          "lon": 126.95645537101001,
          "pop": 192.0
        },
        {
          "lat": 37.49175206792885,
          "lon": 126.95644883946322,
          "pop": 0.0
        },
        {
          "lat": 37.486349299914785,
          "lon": 126.95761911628747,
          "pop": 253.0
        },
        {
          "lat": 37.487250628111816,
          "lon": 126.95761259978316,
          "pop": 330.0
        },
        {
          "lat": 37.48815195617001,
          "lon": 126.95760608298852,
          "pop": 218.0
        },
        {
          "lat": 37.489053284089394,
          "lon": 126.95759956590364,
          "pop": 150.0
        },
        {
          "lat": 37.48995461186995,
          "lon": 126.95759304852847,
          "pop": 354.0
        },
        {
          "lat": 37.49085593951169,
          "lon": 126.957586530863,
          "pop": 440.0
        },
        {
          "lat": 37.491757267014606,
          "lon": 126.9575800129073,
          "pop": 646.0
        },
        {
          "lat": 37.492658594378675,
          "lon": 126.9575734946612,
          "pop": 0.0
        },
        {
          "lat": 37.48635448716286,
          "lon": 126.95875020848462,
          "pop": 103.0
        },
        {
          "lat": 37.4872558155278,
          "lon": 126.95874370556825,
          "pop": 412.0
        },
        {
          "lat": 37.488157143753945,
          "lon": 126.95873720236229,
          "pop": 213.0
        },
        {
          "lat": 37.48905847184126,
          "lon": 126.95873069886666,
          "pop": 286.0
        },
        {
          "lat": 37.489959799789766,
          "lon": 126.95872419508134,
          "pop": 348.0
        },
        {
          "lat": 37.49086112759944,
          "lon": 126.95871769100634,
          "pop": 396.0
        },
        {
          "lat": 37.49176245527029,
          "lon": 126.95871118664168,
          "pop": 280.0
        },
        {
          "lat": 37.492663782802325,
          "lon": 126.95870468198727,
          "pop": 564.0
        },
        {
          "lat": 37.493565110195526,
          "lon": 126.95869817704317,
          "pop": 0.0
        },
        {
          "lat": 37.48545833491164,
          "lon": 126.95988779001063,
          "pop": 1262.0
        },
        {
          "lat": 37.48635966358299,
          "lon": 126.95988130097135,
          "pop": 454.0
        },
        {
          "lat": 37.48726099211551,
          "lon": 126.95987481164308,
          "pop": 557.0
        },
        {
          "lat": 37.48816232050921,
          "lon": 126.95986832202573,
          "pop": 468.0
        },
        {
          "lat": 37.48906364876411,
          "lon": 126.95986183211936,
          "pop": 688.0
        },
        {
          "lat": 37.48996497688021,
          "lon": 126.95985534192393,
          "pop": 754.0
        },
        {
          "lat": 37.49086630485747,
       

In [48]:
# =========================================================
# [부록 2] GridLayer — 격자 인구를 정사각형 기둥으로 집계
# =========================================================
# 참고: https://deckgl.readthedocs.io/en/latest/gallery/grid_layer.html
#
# - cell_size(m) 단위 정사각 셀로 포인트를 자동 집계
# - HexagonLayer와 달리 직교 격자 → 행정 격자 데이터와 시각적으로 잘 어울림
# - upperPercentile: 상위 N% 이상만 색상 표시 (극단치 제거 효과)

grid_layer = pdk.Layer(
    "GridLayer",
    data=df_pts,                               # 포인트 DataFrame
    get_position=["lon", "lat"],               # 위치 컬럼
    cell_size=200,                             # 셀 한 변 길이 (m) — 원본 격자 100m보다 넓게 집계
    elevation_scale=6,                         # 높이 배율
    elevation_range=[0, 600],                  # 최소/최대 높이
    extruded=True,                             # 3D 돌출 ON
    pickable=True,                             # 클릭 상호작용
    auto_highlight=True,                       # 마우스오버 강조
    upper_percentile=100,                      # 상위 100% 모두 표시 (필요 시 99 등으로 줄여 극단치 제거)
    coverage=0.92,                             # 셀 대비 기둥 비율 (0~1, 여백 효과)
    color_range=[                              # 6단계 색상 (연보라 → 진보라/빨강)
        [240, 240, 255, 220],
        [180, 180, 255, 220],
        [120, 120, 255, 220],
        [80,  80,  220, 220],
        [200, 60,  100, 220],
        [150, 10,  50,  230],
    ],
    get_elevation_weight="pop",                # 높이 집계 기준: 인구
    elevation_aggregation="SUM",               # 집계 방식: 합계
    get_color_weight="pop",                    # 색상 집계 기준: 인구
    color_aggregation="SUM",                   # 집계 방식: 합계
)

view_grid = pdk.ViewState(
    latitude=float(center_ll_pts["lat"]),
    longitude=float(center_ll_pts["lon"]),
    zoom=13,
    pitch=45,
    bearing=30,                                # 약간 돌려서 격자 깊이감 강조
)

deck_grid = pdk.Deck(
    layers=[grid_layer],
    initial_view_state=view_grid,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",  # 어두운 배경
    tooltip={"text": "집계 인구: {elevationValue}\n포함 격자 수: {colorValue}"},
)

# ── Jupyter 인라인 렌더링 ──────────────────────────────────
deck_grid

# ── HTML로 저장하려면 아래 주석 해제 ─────────────────────
# deck_grid.to_html(os.path.join(OUT_DIR, "pydeck_grid.html"), open_browser=False)
# print("저장:", os.path.join(OUT_DIR, "pydeck_grid.html"))

{
  "initialViewState": {
    "bearing": 30,
    "latitude": 37.46670139512002,
    "longitude": 126.9757092765704,
    "pitch": 45,
    "zoom": 13
  },
  "layers": [
    {
      "@@type": "GridLayer",
      "autoHighlight": true,
      "cellSize": 200,
      "colorAggregation": "@@=SUM",
      "colorRange": [
        [
          240,
          240,
          255,
          220
        ],
        [
          180,
          180,
          255,
          220
        ],
        [
          120,
          120,
          255,
          220
        ],
        [
          80,
          80,
          220,
          220
        ],
        [
          200,
          60,
          100,
          220
        ],
        [
          150,
          10,
          50,
          230
        ]
      ],
      "coverage": 0.92,
      "data": [
        {
          "lat": 37.48634410183875,
          "lon": 126.95648802438059,
          "pop": 63.0
        },
        {
          "lat": 37.489949413120776,
          "lon": 126.95646190226587,
          "pop": 864.0
        },
        {
          "lat": 37.49085074059423,
          "lon": 126.95645537101001,
          "pop": 192.0
        },
        {
          "lat": 37.49175206792885,
          "lon": 126.95644883946322,
          "pop": 0.0
        },
        {
          "lat": 37.486349299914785,
          "lon": 126.95761911628747,
          "pop": 253.0
        },
        {
          "lat": 37.487250628111816,
          "lon": 126.95761259978316,
          "pop": 330.0
        },
        {
          "lat": 37.48815195617001,
          "lon": 126.95760608298852,
          "pop": 218.0
        },
        {
          "lat": 37.489053284089394,
          "lon": 126.95759956590364,
          "pop": 150.0
        },
        {
          "lat": 37.48995461186995,
          "lon": 126.95759304852847,
          "pop": 354.0
        },
        {
          "lat": 37.49085593951169,
          "lon": 126.957586530863,
          "pop": 440.0
        },
        {
          "lat": 37.491757267014606,
          "lon": 126.9575800129073,
          "pop": 646.0
        },
        {
          "lat": 37.492658594378675,
          "lon": 126.9575734946612,
          "pop": 0.0
        },
        {
          "lat": 37.48635448716286,
          "lon": 126.95875020848462,
          "pop": 103.0
        },
        {
          "lat": 37.4872558155278,
          "lon": 126.95874370556825,
          "pop": 412.0
        },
        {
          "lat": 37.488157143753945,
          "lon": 126.95873720236229,
          "pop": 213.0
        },
        {
          "lat": 37.48905847184126,
          "lon": 126.95873069886666,
          "pop": 286.0
        },
        {
          "lat": 37.489959799789766,
          "lon": 126.95872419508134,
          "pop": 348.0
        },
        {
          "lat": 37.49086112759944,
          "lon": 126.95871769100634,
          "pop": 396.0
        },
        {
          "lat": 37.49176245527029,
          "lon": 126.95871118664168,
          "pop": 280.0
        },
        {
          "lat": 37.492663782802325,
          "lon": 126.95870468198727,
          "pop": 564.0
        },
        {
          "lat": 37.493565110195526,
          "lon": 126.95869817704317,
          "pop": 0.0
        },
        {
          "lat": 37.48545833491164,
          "lon": 126.95988779001063,
          "pop": 1262.0
        },
        {
          "lat": 37.48635966358299,
          "lon": 126.95988130097135,
          "pop": 454.0
        },
        {
          "lat": 37.48726099211551,
          "lon": 126.95987481164308,
          "pop": 557.0
        },
        {
          "lat": 37.48816232050921,
          "lon": 126.95986832202573,
          "pop": 468.0
        },
        {
          "lat": 37.48906364876411,
          "lon": 126.95986183211936,
          "pop": 688.0
        },
        {
          "lat": 37.48996497688021,
          "lon": 126.95985534192393,
          "pop": 754.0
        },
       